## 准备数据

In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [2]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [3]:
class myModel:
    def __init__(self):
        ####################
        '''声明模型对应的参数'''
        ####################
        # 输入层到隐藏层的参数 (784 -> 256)
        self.W1 = tf.Variable(tf.random.truncated_normal([784, 256], stddev=0.1))
        self.b1 = tf.Variable(tf.zeros([256]))
        # 隐藏层到输出层的参数 (256 -> 10)
        self.W2 = tf.Variable(tf.random.truncated_normal([256, 10], stddev=0.1))
        self.b2 = tf.Variable(tf.zeros([10]))
        
    def __call__(self, x):
        ####################
        '''实现模型函数体，返回未归一化的logits'''
        ####################
        # 将输入展平为 (batch_size, 784)
        x = tf.reshape(x, [-1, 784])
        # 第一层：线性变换 + ReLU激活
        h1 = tf.nn.relu(tf.matmul(x, self.W1) + self.b1)
        # 第二层：线性变换（输出logits，不需要激活函数）
        logits = tf.matmul(h1, self.W2) + self.b2
        return logits
        
model = myModel()

optimizer = optimizers.Adam()

## 计算 loss

In [4]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    for g, v in zip(grads, trainable_vars):
        v.assign_sub(0.01*g)

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [5]:
train_data, test_data = mnist_dataset()
for epoch in range(50):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 260s 23us/step
epoch 0 : loss 2.4417777 ; accuracy 0.122083336
epoch 1 : loss 2.4219918 ; accuracy 0.12683333
epoch 2 : loss 2.4029918 ; accuracy 0.13225
epoch 3 : loss 2.384664 ; accuracy 0.1371
epoch 4 : loss 2.3669176 ; accuracy 0.14341667
epoch 5 : loss 2.3496802 ; accuracy 0.14921667
epoch 6 : loss 2.3328924 ; accuracy 0.15585
epoch 7 : loss 2.3165038 ; accuracy 0.16255
epoch 8 : loss 2.3004699 ; accuracy 0.169
epoch 9 : loss 2.2847562 ; accuracy 0.17533334
epoch 10 : loss 2.269333 ; accuracy 0.1823
epoch 11 : loss 2.2541745 ; accuracy 0.19061667
epoch 12 : loss 2.239263 ; accuracy 0.19786666
epoch 13 : loss 2.22458 ; accuracy 0.2061
epoch 14 : loss 2.2101097 ; accuracy 0.21368334
epoch 15 : loss 2.19584 ; accuracy 0.22173333
epoch 16 : loss 2.1817584 ; accuracy 0.22948334
epoch 17 : loss 2.1678562 ; accuracy 0.23758334
epoch 18 : loss 2.1541247 ; accuracy 0.2457
epoch 19 : loss 2.1405556 ; accuracy 0.25396666
epoch 20 : loss 2.1271424 ; accu